In [23]:
import pandas as pd
import pyarrow
from itertools import combinations
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
import scipy.optimize as sco
from scipy.interpolate import make_interp_spline

# Configuración de Paleta de Colores BBVA
paleta_bbva = {
    "azul_logotipo": "#14549C",
    "azul_oscuro_digital": "#06257C",
    "azul_celeste_aqua": "#00B1E1"
}
colores_bbva_list = ["#14549C", "#06257C", "#00B1E1"]

In [24]:
from pathlib import Path
import pandas as pd

# BASE_DIR apunta a la carpeta CODIGO
BASE_DIR = Path.cwd()

# Carga df_diario.parquet desde la raíz del proyecto (compatible con GitHub)
ruta_archivo = BASE_DIR.parent / "df_diario.parquet"

df = pd.read_parquet(ruta_archivo)
df.head()

Ticker           AAPL                                                AMGN  \
Atributo        close       high        low       open volume       close   
date                                                                        
2018-06-06  45.890783  45.890783  45.405855  45.803260    0.0  115.655362   
2018-06-07  45.772508  45.928631  45.508754  45.927449    0.0  115.497849   
2018-06-08  45.301773  45.416500  44.898454  45.176401    0.0  116.127938   
2018-06-11  45.218980  45.410586  44.998988  45.262742    0.0  116.323276   
2018-06-12  45.493378  45.559613  45.222528  45.272204    0.0  116.613127   

Ticker                                                     ...         UNH  \
Atributo          high         low        open     volume  ...       close   
date                                                       ...               
2018-06-06  116.033418  113.966681  114.684998  2213100.0  ...  193.867722   
2018-06-07  115.970426  115.000062  115.604965  2729600.0  ...  194.125062   
2018-06-08  116.978571  115.686868  115.869594  2865000.0  ...  195.450467   
2018-06-11  116.455602  115.264699  116.027127  2409400.0  ...  197.867518   
2018-06-12  116.644625  115.567154  116.109043  2045300.0  ...  197.711575   

Ticker                                                              V  \
Atributo          high         low        open     volume       close   
date                                                                    
2018-06-06  193.922305  191.123252  192.051075  2650300.0  122.520808   
2018-06-07  194.920341  193.555898  194.733213  2117200.0  120.327169   
2018-06-08  196.089811  193.415510  194.499267  3034600.0  121.136333   
2018-06-11  198.810938  195.232192  195.575254  2775200.0  128.074994   
2018-06-12  197.992261  195.052859  197.415292  2567200.0  128.341777   

Ticker                                                     
Atributo          high         low        open     volume  
date                                                       
2018-06-06  122.547778  120.120380  120.471004  8065800.0  
2018-06-07  122.790537  119.616936  122.592748  8309200.0  
2018-06-08  121.298153  119.778778  119.967582  5261000.0  
2018-06-11  128.784828  127.808211  128.437057        0.0  
2018-06-12  128.713368  127.236533  127.660527        0.0  

[5 rows x 130 columns]

In [25]:
# Extraer y limpiar nombres de columnas
df_close = df[[c for c in df.columns if c[1] == "close"]].copy()
df_close.columns = [c[0] for c in df_close.columns if c[1] == "close"]

# Graficar
fig = px.line(
    df_close, title="Evolución del Precio de Cierre", labels={"value": "Precio"},
    color_discrete_sequence=colores_bbva_list
)
fig.show()

In [26]:
# 1. Preparar retornos
activos = ['AXP', 'CRM', 'MSFT', 'KO', 'JNJ', 'GS', 'PG', 'DIS', 'UNH', 'MRK', 'V', 'CVX', 'BA', 'INTC', 'MMM', 'IBM', 'CSCO', 'AAPL', 'MCD', 'CAT', 'HD', 'AMGN', 'JPM', 'NKE', 'TRV', 'DOW']
retornos = df.xs("close", axis=1, level=1).filter(items=activos).pct_change().dropna()

r_anual = retornos.mean() * 252
cov_anual = retornos.cov() * 252
matriz_corr = retornos.corr()

# 2. Calcular métricas para cada trío
resultados = []
for trio in combinations(retornos.columns, 3):
    t = list(trio)
    rent = r_anual[t].mean()
    vol = np.sqrt(cov_anual.loc[t, t].values.sum() / 9)
    corr_media = (matriz_corr.loc[t, t].values.sum() - 3) / 6
    
    resultados.append({
        "Trio": " - ".join(t), 
        "Corr": round(corr_media, 4), 
        "Sharpe": round(rent / vol, 4), 
        "Rent": rent, 
        "Vol": vol
    })

df_t = pd.DataFrame(resultados)

# 3. Identificar el mejor trío
top = df_t.loc[(df_t["Sharpe"] - df_t["Corr"]).idxmax()]
opt_label = f"ÓPTIMO: {top['Trio']}"
df_t["Cat"] = np.where(df_t.index == top.name, opt_label, "Otros")

# 4. Crear gráfico
fig = px.scatter(
    df_t, x="Corr", y="Sharpe", color="Cat", hover_data=["Trio", "Rent", "Vol"],
    color_discrete_map={"Otros": paleta_bbva["azul_celeste_aqua"], opt_label: paleta_bbva["azul_oscuro_digital"]},
    title="Sharpe vs Correlación",
    labels={"Corr": "Correlación Media", "Sharpe": "Ratio Sharpe"}
)

fig.add_annotation(
    x=top["Corr"], y=top["Sharpe"], 
    text=f"<b>{top['Trio']}</b><br>Sharpe: {top['Sharpe']:.2f} | Corr: {top['Corr']:.2f}",
    showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_logotipo"], ax=-80, ay=-30, font=dict(color=paleta_bbva["azul_logotipo"])
)

fig.update_traces(marker=dict(size=5, opacity=0.4), selector=dict(name="Otros"))
fig.update_traces(marker=dict(size=14, symbol="star"), selector=dict(name=opt_label))
fig.update_layout(height=700, width=850).show()

C:\Users\nenek\AppData\Local\Temp\ipykernel_25976\4142705081.py:3: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [27]:
cartera_tickers = ['PG', 'IBM', 'GS']
matriz_corr = retornos[cartera_tickers].corr().round(4)
print(matriz_corr)

         PG     IBM      GS
PG   1.0000  0.4484  0.1307
IBM  0.4484  1.0000  0.2938
GS   0.1307  0.2938  1.0000


In [28]:
# 1. Preparar datos de retornos
df_close = df.xs("close", axis=1, level=1).filter(items=activos)
df_close.index = pd.to_datetime(df[("date", "date")] if ("date", "date") in df.columns else df.get("date", df.index))
ret = df_close.pct_change().dropna()

r_d, cov_d, corr_m = ret.mean(), ret.cov(), ret.corr()
w = np.array([1/3, 1/3, 1/3])
resultados = []

# 2. Filtrar tríos con correlación asimétrica
for trio in combinations(ret.columns, 3):
    t = list(trio)
    c_sub = corr_m.loc[t, t].values
    r12, r13, r23 = c_sub[0, 1], c_sub[0, 2], c_sub[1, 2]
    _, r_min2, r_max = sorted([r12, r13, r23])
    
    if r_max >= 0.50 and r_min2 <= 0.25:
        rent = np.dot(r_d[t], w) * 252
        vol = np.sqrt(w.T @ cov_d.loc[t, t] @ w) * np.sqrt(252)
        
        if r_max == r12: bloque = f"{t[0]}-{t[1]} + {t[2]}"
        elif r_max == r13: bloque = f"{t[0]}-{t[2]} + {t[1]}"
        else: bloque = f"{t[1]}-{t[2]} + {t[0]}"
            
        resultados.append({
            "Trio": " - ".join(t), "Estructura": bloque,
            "Corr_Max_Par": round(r_max, 4), "Corr": round((r12 + r13 + r23) / 3, 4),
            "Sharpe": round(rent / vol, 4), "Rent": rent, "Vol": vol
        })

df_t = pd.DataFrame(resultados)

# 3. Identificar el óptimo y graficar
top = df_t.loc[df_t["Sharpe"].idxmax()]
opt_cat = f"ÓPTIMO ASIMÉTRICO: {top['Trio']}"
df_t["Cat"] = np.where(df_t.index == top.name, opt_cat, "Otros Asimétricos")

fig = px.scatter(
    df_t, x="Corr", y="Sharpe", color="Cat", hover_data=["Trio", "Estructura", "Corr_Max_Par", "Rent", "Vol"],
    color_discrete_map={"Otros Asimétricos": paleta_bbva["azul_celeste_aqua"], opt_cat: paleta_bbva["azul_oscuro_digital"]},
    title="Estrategia Asimétrica: Sharpe vs Correlación",
    labels={"Corr": "Correlación Media", "Sharpe": "Ratio Sharpe"}
)

fig.add_annotation(
    x=top["Corr"], y=top["Sharpe"], 
    text=f"<b>{top['Trio']}</b><br>{top['Estructura']}<br>Sharpe: {top['Sharpe']:.2f} | Corr Media: {top['Corr']:.2f}",
    showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_logotipo"], ax=-120, ay=-40, font=dict(color=paleta_bbva["azul_logotipo"])
)

fig.update_traces(marker=dict(size=5, opacity=0.4), selector=dict(name="Otros Asimétricos"))
fig.update_traces(marker=dict(size=14, symbol="star"), selector=dict(name=opt_cat))
fig.update_layout(height=700, width=850).show()

C:\Users\nenek\AppData\Local\Temp\ipykernel_25976\2205601181.py:4: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [29]:
cartera_tickers = ['KO', 'CAT', 'GS']
matriz_corr = ret[cartera_tickers].corr().round(4)
print(matriz_corr)

         KO     CAT      GS
KO   1.0000  0.1591  0.2110
CAT  0.1591  1.0000  0.6216
GS   0.2110  0.6216  1.0000


In [30]:
# 1. SELECCIÓN DE ACTIVOS Y PARÁMETROS
activos_3 = ['MSFT', 'JNJ', 'JPM']
precios_3 = df.xs("close", axis=1, level=1)[activos_3].dropna()
rent_activos = precios_3.pct_change().dropna()

r_mean = rent_activos.mean() * 252
cov_mat = rent_activos.cov() * 252
rf, rf_diario = 0.03, 0.03 / 252
n_assets = len(activos_3)

# 2. FUNCIONES DE EVALUACIÓN Y OPTIMIZACIÓN

def portfolio_performance_sortino(weights, returns):
    weights = np.array(weights)
    ret_p = np.sum(r_mean * weights)
    
    port_returns_daily = returns @ weights
    downside_returns = np.minimum(0, port_returns_daily - rf_diario)
    downside_std = np.sqrt(np.mean(downside_returns**2)) * np.sqrt(252)
    
    sortino = 0 if downside_std == 0 else (ret_p - rf) / downside_std
    vol_p = np.sqrt(weights.T @ cov_mat @ weights)
    
    return vol_p, ret_p, sortino, downside_std

bounds = tuple((0, 1) for _ in range(n_assets))
init_guess = [1.0 / n_assets] * n_assets
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Optimizaciones
opt_vol = sco.minimize(lambda w: portfolio_performance_sortino(w, rent_activos)[0], init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
vol_min, ret_min, _, _ = portfolio_performance_sortino(opt_vol.x, rent_activos)

opt_sortino = sco.minimize(lambda w: -portfolio_performance_sortino(w, rent_activos)[2], init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
vol_max_sortino, ret_max_sortino, sortino_max, _ = portfolio_performance_sortino(opt_sortino.x, rent_activos)

# 3. SIMULACIÓN MONTE CARLO Y FRONTERA

np.random.seed(42)
weights_sim = np.random.dirichlet(np.ones(n_assets), size=10000)

sim_daily_returns = rent_activos.values @ weights_sim.T
sim_rets = weights_sim @ r_mean.values
sim_vols = np.sqrt(np.einsum('ij,jk,ik->i', weights_sim, cov_mat.values, weights_sim))

downside_diff = np.minimum(0, sim_daily_returns - rf_diario)
sim_downside_std = np.sqrt(np.mean(downside_diff**2, axis=0)) * np.sqrt(252)
sim_sortinos = np.where(sim_downside_std > 0, (sim_rets - rf) / sim_downside_std, 0)

target_rets = np.linspace(ret_min, r_mean.max(), 50)
frontier_vols = []
for t in target_rets:
    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w: np.sum(r_mean * w) - t})
    res = sco.minimize(lambda w: portfolio_performance_sortino(w, rent_activos)[0], init_guess, method='SLSQP', bounds=bounds, constraints=cons)
    frontier_vols.append(res.fun)

# 4. VISUALIZACIÓN

fig = go.Figure()

fig.add_trace(go.Scatter(x=sim_vols, y=sim_rets, mode='markers',
    marker=dict(color=sim_sortinos, colorscale=[[0, paleta_bbva["azul_celeste_aqua"]], [1, paleta_bbva["azul_oscuro_digital"]]], size=4, showscale=True, colorbar=dict(title="Ratio Sortino")),
    name='Simulación', opacity=0.5))

fig.add_trace(go.Scatter(x=frontier_vols, y=target_rets, mode='lines',
    line=dict(color=paleta_bbva["azul_oscuro_digital"], width=2.5, dash='dash'), name='Frontera Eficiente'))

fig.add_trace(go.Scatter(x=np.sqrt(np.diag(cov_mat)), y=r_mean.values, mode='markers+text',
    marker=dict(color=paleta_bbva["azul_logotipo"], size=10), text=activos_3, textposition="top center", name='Activos'))

fig.add_trace(go.Scatter(x=[vol_min], y=[ret_min], mode='markers',
    marker=dict(color=paleta_bbva["azul_celeste_aqua"], size=14, symbol='triangle-up'), name=f'Mínima Varianza: {vol_min:.1%}'))

fig.add_trace(go.Scatter(x=[vol_max_sortino], y=[ret_max_sortino], mode='markers',
    marker=dict(color=paleta_bbva["azul_oscuro_digital"], size=15, symbol='triangle-up'), name=f'Máx. Sortino: {sortino_max:.2f}'))

pesos_str = "<br>".join([f"• {a}: {p:.1%}" for a, p in zip(activos_3, opt_sortino.x)])
fig.add_annotation(x=vol_max_sortino, y=ret_max_sortino,
    text=f"<b>Cartera Óptima</b><br>{pesos_str}",
    showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_oscuro_digital"], ax=100, ay=-40, bgcolor="white", bordercolor="black")

fig.update_layout(
    title="<b>Optimización de Cartera (Ratio de Sortino)</b>",
    xaxis=dict(title="Volatilidad Anualizada", tickformat=".1%"),
    yaxis=dict(title="Rentabilidad Esperada Anualizada", tickformat=".1%"),
    template="plotly_white", height=650, width=950,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

In [31]:
# 1. Datos y retornos
df_close = df.xs("close", axis=1, level=1)[lambda x: x.columns.intersection(activos)].dropna()
ret = df_close.pct_change().dropna()
r_d, cov_d = ret.mean() * 252, ret.cov() * 252
rf, rf_d, w_eq = 0.03, 0.03 / 252, np.array([1/3, 1/3, 1/3])

# 2. Métricas de los tríos
res = []
for trio in combinations(ret.columns, 3):
    t_l = list(trio)
    r_c = np.dot(r_d[t_l], w_eq)
    v_c = np.sqrt(cov_d.loc[t_l, t_l].values.sum() / 9)
    port_ret = ret[t_l].mean(axis=1)
    
    down_std = np.sqrt(np.mean(np.minimum(0, port_ret - rf_d)**2)) * np.sqrt(252)
    sortino = (r_c - rf) / down_std if down_std > 0 else 0
    
    res.append({"Trio": " - ".join(t_l), "Vol": v_c, "Rent": r_c, "Sortino": sortino})

df_trios = pd.DataFrame(res)
top_s, top_v = df_trios.loc[df_trios["Sortino"].idxmax()], df_trios.loc[df_trios["Vol"].idxmin()]

# 3. Frontera eficiente y spline
df_ef = df_trios[(df_trios["Rent"] >= top_v["Rent"]) & (df_trios["Rent"] <= top_s["Rent"])]
df_ef['Bin'] = pd.qcut(df_ef['Rent'], q=12, duplicates='drop')
nodos = df_ef.groupby('Bin', observed=True).apply(lambda g: g.loc[g['Vol'].idxmin()]).dropna()

f_rets = np.concatenate(([top_v["Rent"]], nodos['Rent'].values, [top_s["Rent"]]))
f_vols = np.concatenate(([top_v["Vol"]], nodos['Vol'].values, [top_s["Vol"]]))
orden = np.argsort(f_rets)
f_rets, f_vols = f_rets[orden], f_vols[orden]

f_v_lim, f_r_lim = [f_vols[0]], [f_rets[0]]
for vr, rt in zip(f_vols[1:], f_rets[1:]):
    if rt > f_r_lim[-1]:
        f_v_lim.append(vr); f_r_lim.append(rt)

spl = make_interp_spline(f_r_lim, f_v_lim, k=min(2, len(f_r_lim)-1))
s_rets = np.linspace(min(f_r_lim), max(f_r_lim), 150)

# 4. Gráfico
fig = go.Figure([
    go.Scatter(x=df_trios["Vol"], y=df_trios["Rent"], mode='markers', name='Tríos (1/3)', opacity=0.4,
               marker=dict(color=df_trios["Sortino"], colorscale=[[0, paleta_bbva["azul_celeste_aqua"]], [1, paleta_bbva["azul_oscuro_digital"]]], size=4, showscale=True, colorbar=dict(title="Sortino")),
               text=df_trios["Trio"], hovertemplate="<b>%{text}</b><br>Vol: %{x:.2%}<br>Rent: %{y:.2%}<br>Sortino: %{marker.color:.2f}<extra></extra>"),
    go.Scatter(x=spl(s_rets), y=s_rets, mode='lines', name='Frontera Eficiente', line=dict(color=paleta_bbva["azul_logotipo"], width=2.5, dash='dash')),
    go.Scatter(x=[top_v["Vol"]], y=[top_v["Rent"]], mode='markers', name=f'Mín. Var: {top_v["Trio"]}', marker=dict(color=paleta_bbva["azul_celeste_aqua"], size=12, symbol='diamond', line=dict(color='black', width=1))),
    go.Scatter(x=[top_s["Vol"]], y=[top_s["Rent"]], mode='markers', name=f'Máx. Sortino: {top_s["Trio"]}', marker=dict(color=paleta_bbva["azul_oscuro_digital"], size=15, symbol='star', line=dict(color='black', width=1)))
])

fig.add_annotation(x=top_s["Vol"], y=top_s["Rent"], text=f"<b>ÓPTIMO SORTINO: {top_s['Trio']}</b><br>Sortino: {top_s['Sortino']:.2f} | Rent: {top_s['Rent']:.1%} | Vol: {top_s['Vol']:.1%}",
                   showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_oscuro_digital"], ax=140, ay=-40, bgcolor="white", bordercolor="black", borderpad=4)

fig.add_annotation(x=top_v["Vol"], y=top_v["Rent"], text=f"<b>MÍN. VARIANZA: {top_v['Trio']}</b><br>Vol: {top_v['Vol']:.1%} | Rent: {top_v['Rent']:.1%}",
                   showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_celeste_aqua"], ax=-140, ay=-30, bgcolor="white", bordercolor="black", borderpad=4)

fig.update_layout(title="<b>Frontera Eficiente de Tríos (Ratio de Sortino)</b>",
                  xaxis=dict(title="Volatilidad Anualizada", tickformat=".1%", showgrid=True, gridcolor='#E5E5E5'),
                  yaxis=dict(title="Rentabilidad Esperada Anualizada", tickformat=".1%", showgrid=True, gridcolor='#E5E5E5'),
                  template="plotly_white", height=650, width=950,
                  legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)"))
fig.show()

C:\Users\nenek\AppData\Local\Temp\ipykernel_25976\1412284772.py:25: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\nenek\AppData\Local\Temp\ipykernel_25976\1412284772.py:26: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [32]:
# 1. CONFIGURACIÓN
WINDOW_SIZE = 80
rf_diario = 0.03 / 252
w_eq = np.array([1/3, 1/3, 1/3])

r_d, cov_d = ret.mean() * 252, ret.cov() * 252

# 2. CORRELACIONES Y FILTRADO
rolling_corr = ret.rolling(WINDOW_SIZE).corr().dropna()
corr_mean_80d = rolling_corr.groupby(level=1).mean()

fechas_covid = [f for f in rolling_corr.index.get_level_values(0).unique() if '2020-02-19' <= str(f) <= '2020-11-09']
UMBRAL_ALTO, UMBRAL_BAJO = 0.45, 0.35

# 3. EVALUACIÓN DE COMBINACIONES
candidatas = []

for t in combinations(ret.columns, 3):
    t_l = list(t)
    p_means = sorted([{"pair": (t_l[i], t_l[j]), "name": f"{t_l[i]} - {t_l[j]}", "mean_80d": corr_mean_80d.loc[t_l[i], t_l[j]]} 
                      for i, j in [(0,1), (0,2), (1,2)]], key=lambda x: x["mean_80d"])
    
    # Criterio 2: Estructura de correlación
    if not (p_means[2]["mean_80d"] >= UMBRAL_ALTO and p_means[1]["mean_80d"] < UMBRAL_BAJO and p_means[0]["mean_80d"] < UMBRAL_BAJO):
        continue
        
    # Criterio 3: Mínimos en COVID
    s_covid = [[rolling_corr.loc[(f, p["pair"][0]), p["pair"][1]] for f in fechas_covid if (f, p["pair"][0]) in rolling_corr.index] for p in p_means[:2]]
    if not all(s_covid) or any(len(s) == 0 for s in s_covid):
        continue
        
    min_c1, min_c2 = np.min(s_covid[0]), np.min(s_covid[1])
    
    # Criterio 1: Métricas de cartera
    r_c = np.dot(r_d[t_l], w_eq)
    v_c = np.sqrt(cov_d.loc[t_l, t_l].values.sum() / 9)
    
    downside = np.minimum(0, ret[t_l].mean(axis=1) - rf_diario)
    sortino = (r_c - 0.03) / (np.sqrt(np.mean(downside**2)) * np.sqrt(252)) if np.any(downside < 0) else 0
    
    candidatas.append({
        "Trio": " - ".join(t_l), "Rent": r_c, "Vol": v_c, "Sortino": sortino,
        "Par_Alto": p_means[2], "Par_Bajo1": {**p_means[0], "min_covid": min_c1},
        "Par_Bajo2": {**p_means[1], "min_covid": min_c2}, "Score_Crisis": (min_c1 + min_c2) / 2
    })

df_cand = pd.DataFrame(candidatas)

# 4. FRONTERA EFICIENTE
if df_cand.empty:
    print("No se encontraron carteras que cumplan los criterios.")
else:
    top_var = df_cand.loc[df_cand["Vol"].idxmin()]
    df_eficiente = df_cand[df_cand["Rent"] >= top_var["Rent"]].copy()
    
    df_eficiente['Bin'] = pd.qcut(df_eficiente['Rent'], q=min(8, len(df_eficiente)), duplicates='drop')
    frontera = df_eficiente.groupby('Bin', observed=True).apply(lambda g: g.loc[g['Sortino'].idxmax()]).drop_duplicates(subset=['Trio']).sort_values(by="Score_Crisis")

    # 5. RESULTADOS 
    print("=" * 90)
    print(" SELECCIÓN DE CARTERAS (VENTANA MÓVIL 80 DÍAS)")
    print("=" * 90)
    print(f"\nCarteras óptimas encontradas: {len(frontera)}\n")

    for idx, (_, row) in enumerate(frontera.iterrows(), 1):
        print(f" {idx:02d}. CARTERA: [ {row['Trio']} ]")
        print(f"     • Rentabilidad: {row['Rent']:.2%} | Volatilidad: {row['Vol']:.2%} | Sortino: {row['Sortino']:.2f}")
        print(f"     • Alto (80d)    : {row['Par_Alto']['name']} = {row['Par_Alto']['mean_80d']:.2f}")
        print(f"     • Defensivo 1   : {row['Par_Bajo1']['name']} = {row['Par_Bajo1']['mean_80d']:.2f} (Min COVID: {row['Par_Bajo1']['min_covid']:.2f})")
        print(f"     • Defensivo 2   : {row['Par_Bajo2']['name']} = {row['Par_Bajo2']['mean_80d']:.2f} (Min COVID: {row['Par_Bajo2']['min_covid']:.2f})")
        print(f"     • Score Crisis  : {row['Score_Crisis']:.2f}")
        print("-" * 90)

 SELECCIÓN DE CARTERAS (VENTANA MÓVIL 80 DÍAS)

Carteras óptimas encontradas: 8

 01. CARTERA: [ CAT - JNJ - PG ]
     • Rentabilidad: 16.34% | Volatilidad: 19.26% | Sortino: 0.99
     • Alto (80d)    : JNJ - PG = 0.49
     • Defensivo 1   : CAT - PG = 0.09 (Min COVID: -0.27)
     • Defensivo 2   : CAT - JNJ = 0.16 (Min COVID: -0.14)
     • Score Crisis  : -0.20
------------------------------------------------------------------------------------------
 02. CARTERA: [ AAPL - CAT - MSFT ]
     • Rentabilidad: 39.40% | Volatilidad: 47.56% | Sortino: 1.87
     • Alto (80d)    : AAPL - MSFT = 0.59
     • Defensivo 1   : AAPL - CAT = 0.23 (Min COVID: -0.29)
     • Defensivo 2   : CAT - MSFT = 0.23 (Min COVID: -0.11)
     • Score Crisis  : -0.20
------------------------------------------------------------------------------------------
 03. CARTERA: [ GS - PG - V ]
     • Rentabilidad: 19.49% | Volatilidad: 19.82% | Sortino: 1.18
     • Alto (80d)    : GS - V = 0.46
     • Defensivo 1   : GS -

C:\Users\nenek\AppData\Local\Temp\ipykernel_25976\1691599460.py:57: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

